# Formula 1 Race Strategy Simulator
## Notebook 01: Physics & Synthetic Model Exploration

This notebook explores the analytical physics engine developed in **Phase 1**:
1. **Lap-Time Decomposition**: Breaking down a lap into base track pace, fuel burn, tyre degradation, traffic, and pit loss.
2. **Non-Linear Tyre Degradation**: Modeling quadratic wear $\alpha a + \beta a^2$ and exponential thermal cliff $\gamma e^{\kappa (a - L_{cliff})}$.
3. **Fuel Burn Dynamics**: Mass depletion $m_{fuel}(l) = m_0 - \beta_{burn} \cdot l$ and acceleration penalty.
4. **Deterministic Race Simulation**: Simulating and comparing 1-stop vs 2-stop strategies at the Bahrain Grand Prix.

In [ ]:
import sys
from pathlib import Path

# Add repository root to Python path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.model import CircuitConfig, TyreCompound, FuelModel, PitStopModel, RaceModel
from src.strategies import Strategy, Stint
from src.simulation import simulate_race
from src.config import BAHRAIN_CONFIG, DEFAULT_COMPOUNDS

sns.set_theme(style="darkgrid")
print("Environment and modules successfully initialized.")

### 1. Non-Linear Tyre Degradation Mechanics

Tyre degradation is modeled with two distinct regimes:
1. **Progressive Structural & Thermal Wear**:
   $$\Delta T_{wear}(a) = \alpha a + \beta a^2$$
2. **Thermal Degradation Cliff**:
   $$\Delta T_{cliff}(a) = \gamma \exp\big(\kappa (a - L_{cliff})\big) \quad \text{for } a > L_{cliff}$$

Let us plot the degradation curves across Soft, Medium, and Hard compounds over 40 laps.

In [ ]:
laps = np.arange(1, 41)
fig, ax = plt.subplots(figsize=(10, 5))

for compound_name, compound in DEFAULT_COMPOUNDS.items():
    wear = [compound.degradation_at_lap(a) for a in laps]
    ax.plot(laps, wear, label=f"{compound.name} (base {compound.base_time_delta:+.2f}s)", linewidth=2.5)

ax.set_title("Non-Linear Tyre Compound Degradation vs Tyre Age", fontsize=14, fontweight="bold")
ax.set_xlabel("Tyre Age (Laps)", fontsize=12)
ax.set_ylabel("Degradation Lap Time Penalty (s)", fontsize=12)
ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### 2. Fuel Mass Depletion & Weight Advantage

As fuel burns at $\sim 1.84\text{ kg/lap}$, the vehicle sheds mass:
$$m_{fuel}(l) = m_0 - \beta_{burn} \cdot (l - 1), \quad \Delta T_{fuel}(l) = \lambda_{fuel} \cdot m_{fuel}(l)$$

This weight reduction provides a significant lap-time gain (over $3.0\text{s}$ across a Grand Prix).

In [ ]:
model = RaceModel(BAHRAIN_CONFIG)
total_laps = model.circuit.total_laps
lap_indices = np.arange(1, total_laps + 1)

fuel_masses = [model.fuel_model.fuel_mass_at_lap(l) for l in lap_indices]
fuel_penalties = [model.fuel_model.lap_time_penalty(m) for m in fuel_masses]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.plot(lap_indices, fuel_masses, color="crimson", linewidth=2)
ax1.set_title("Fuel Mass Depletion Over Race Distance", fontweight="bold")
ax1.set_xlabel("Lap Number")
ax1.set_ylabel("Remaining Fuel (kg)")

ax2.plot(lap_indices, fuel_penalties, color="navy", linewidth=2)
ax2.set_title("Fuel Lap-Time Penalty Evolution", fontweight="bold")
ax2.set_xlabel("Lap Number")
ax2.set_ylabel("Time Penalty (s)")

plt.tight_layout()
plt.show()

### 3. Deterministic Race Simulation: 1-Stop vs 2-Stop

We compare two standard tactical approaches for the 57-lap Bahrain Grand Prix:
* **Strategy A (1-Stop)**: Soft (Laps 1-18) $\to$ Hard (Laps 19-57).
* **Strategy B (2-Stop)**: Soft (Laps 1-15) $\to$ Medium (Laps 16-36) $\to$ Medium (Laps 37-57).

In [ ]:
strat_1stop = Strategy(
    stints=[
        Stint(compound=DEFAULT_COMPOUNDS["Soft"], laps=18),
        Stint(compound=DEFAULT_COMPOUNDS["Hard"], laps=39),
    ],
    name="1-Stop (S -> H)"
)

strat_2stop = Strategy(
    stints=[
        Stint(compound=DEFAULT_COMPOUNDS["Soft"], laps=15),
        Stint(compound=DEFAULT_COMPOUNDS["Medium"], laps=21),
        Stint(compound=DEFAULT_COMPOUNDS["Medium"], laps=21),
    ],
    name="2-Stop (S -> M -> M)"
)

res_1stop = simulate_race(strat_1stop, model)
res_2stop = simulate_race(strat_2stop, model)

print(f"Strategy 1-Stop Duration: {res_1stop.total_time:.3f} s  ({res_1stop.summary()['formatted_time']})")
print(f"Strategy 2-Stop Duration: {res_2stop.total_time:.3f} s  ({res_2stop.summary()['formatted_time']})")
delta = res_1stop.total_time - res_2stop.total_time
winner = "2-Stop" if delta > 0 else "1-Stop"
print(f"Delta: {abs(delta):.3f} s ({winner} is faster)")

### 4. Lap-by-Lap Telemetry Profiling

Let us visualize the effective lap times over the 57 laps, highlighting the pit stop transit losses and compound degradation differences.

In [ ]:
laps_1 = [lr.lap_number for lr in res_1stop.lap_records]
times_1 = [lr.effective_lap_time for lr in res_1stop.lap_records]

laps_2 = [lr.lap_number for lr in res_2stop.lap_records]
times_2 = [lr.effective_lap_time for lr in res_2stop.lap_records]

plt.figure(figsize=(12, 5))
plt.plot(laps_1, times_1, label="1-Stop (S -> H)", color="tab:blue", linewidth=2)
plt.plot(laps_2, times_2, label="2-Stop (S -> M -> M)", color="tab:orange", linewidth=2)

plt.title("Lap-by-Lap Effective Pace Profile: Bahrain Grand Prix", fontsize=14, fontweight="bold")
plt.xlabel("Lap Number", fontsize=12)
plt.ylabel("Effective Lap Time (s)", fontsize=12)
plt.ylim(90, 130)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()